<center style="padding: 1em 2em; margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0; color: rgb(52, 72, 92);">«Обработка данных: текст и время»</h1>
</center>

Продолжим работу с `Pandas`: разберём текстовые столбцы, регулярные выражения, даты и часовые пояса. На примерах поездок и логов научимся извлекать значения, считать длительности и готовить данные к анализу.

## **Данные о поездках**

**С чего начнём?** `read_csv` принимает не только путь к файлу, но и адрес `CSV` в интернете. Используем ссылку, которую вы можете открыть отдельно. В таблице есть момент посадки `pickup`, высадки `dropoff`, расстояние в милях и суммы в долларах.

Возьмём [учебную выборку `taxis` из `seaborn`](https://github.com/mwaskom/seaborn-data/blob/master/taxis.csv). В ней 6433 поездки, а исходные записи происходят из [NYC Taxi & Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page). Это **выборка для примеров**, а не все поездки города. `CSV` прочитаем прямо по ссылке; для запуска понадобится интернет.

---

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxi = pd.read_csv(url)
taxi[["pickup", "dropoff", "distance", "fare", "payment"]].head(3)

,pickup,dropoff,distance,fare,payment
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1.60,7.0,credit card
1,2019-03-04 16:11:55,2019-03-04 16:19:00,0.79,5.0,cash
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1.37,7.5,credit card


Посмотрим размер: одна строка соответствует одной поездке. Позже будем считать поездки по дням, поэтому важно знать, с каким числом записей начали.

In [2]:
taxi.shape

(6433, 14)

### **Пропуски**

В прошлом уроке мы встретили `NaN` на маленьком примере. Здесь пропуски настоящие. Посмотрим, **в каких** столбцах они есть: пустая оплата и пустое название зоны требуют разных решений.

In [3]:
taxi[["pickup", "dropoff", "payment", "pickup_zone", "dropoff_zone"]].isna().sum()

pickup           0
dropoff          0
payment         44
pickup_zone     26
dropoff_zone    45
dtype: int64

У оплаты пропуск можно увидеть и среди значений. По умолчанию `value_counts()` не показывает пустые записи; `dropna=False` добавляет их в подсчёт.

In [4]:
taxi["payment"].value_counts(dropna=False)

payment
credit card    4577
cash           1812
NaN              44
Name: count, dtype: int64

Если вопрос касается способа оплаты, строки без него лучше убрать **только из этого расчёта**. `dropna(subset=["payment"])` оставит строки с известной оплатой. Исходную таблицу `taxi` сохраним для анализа времени и расстояния.

In [5]:
with_payment = taxi.dropna(subset=["payment"])
with_payment.shape

(6389, 14)

### **Текст в столбце.**

**Начнём с одной строки.** В записи города могут оказаться пробелы по краям и разный регистр букв. Метод `strip()` удаляет крайние пробельные символы; `lower()` переводит буквы в нижний регистр. Применим их по очереди.

In [6]:
city_text = "  New York  "
city_text.strip().lower()

'new york'

У столбца доступ `.str` открывает строковые операции для его значений. Возьмём маленький учебный столбец: в нём видны и лишние пробелы, и пропуск. Укажем `dtype="string"`, чтобы явно выбрать строковый тип с пропуском `pd.NA`.

In [7]:
city_labels = pd.Series(["  New York  ", "NEW YORK", None], dtype="string")
city_labels.str.strip().str.lower()

0    new york
1    new york
2        <NA>
dtype: string

Пропуск остался пропуском: неизвестный город не превратился в строку `"None"`. Для реальной таблицы правило очистки выбирают по смыслу: например, регистр в идентификаторе иногда важен.

Вернёмся к поездкам. Найдём зоны посадки, в названии которых есть `Airport`. `.str.contains` проверяет наличие фрагмента; `regex=False` включает обычный поиск подстроки. `na=False` делает результат для пустого названия равным `False`, чтобы маску можно было использовать при выборе строк.

In [8]:
airport_pickups = taxi["pickup_zone"].str.contains("Airport", na=False, regex=False)
airport_pickups.sum()

np.int64(297)

Маска снова выбирает строки, как в первом уроке. Посмотрим несколько поездок из найденных зон.

In [9]:
taxi.loc[airport_pickups, ["pickup_zone", "distance", "fare"]].head(3)

,pickup_zone,distance,fare
10,LaGuardia Airport,3.90,17.0
22,LaGuardia Airport,9.82,31.5
43,LaGuardia Airport,9.50,28.0


Знакомая маска оставила нужные поездки. Пока мы искали готовое слово; к поиску по устройству строки вернёмся после сводки по группам.

---

## **Группы и сводка.**

В прошлой паре мы группировали по одному столбцу. Теперь спросим: сколько поездок было у каждого цвета такси при каждом способе оплаты? Список имён в `groupby` задаёт сразу два признака группировки.

In [10]:
with_payment.groupby(["color", "payment"]).size()

color   payment    
green   cash            400
        credit card     577
yellow  cash           1412
        credit card    4000
dtype: int64

`size()` считает строки группы. `count()` считает заполненные значения выбранного столбца. Чтобы увидеть различие, возьмём отдельную маленькую таблицу с одной неизвестной стоимостью.

In [11]:
fare_example = pd.DataFrame({
    "payment": ["card", "card", "cash"],
    "fare": [10.0, None, 15.0],
})
fare_example.groupby("payment")["fare"].agg(["size", "count"])

,size,count
payment,,
card,2,1
cash,1,1


У `card` две строки, но только одна известная стоимость. Для количества поездок нам нужен `size`.

**Как получить несколько показателей с понятными названиями?** В именованной агрегации запись `trips=("fare", "size")` означает: назвать результат `trips`, выбрать столбец `fare` и применить `size`. Добавим медианную стоимость тем же способом.

In [12]:
by_color_payment = with_payment.groupby(["color", "payment"]).agg(
    trips=("fare", "size"),
    median_fare=("fare", "median"),
)
by_color_payment

trips  median_fare
color  payment                        
green  cash           400          8.0
       credit card    577         12.5
yellow cash          1412          8.5
       credit card   4000          9.5

Слева теперь два уровня индекса: цвет и способ оплаты. Это **`MultiIndex`** – строку определяет пара подписей. Если дальше нужна обычная таблица со столбцами `color` и `payment`, используем `reset_index()`.

In [13]:
by_color_payment.reset_index()

,color,payment,trips,median_fare
0,green,cash,400,8.0
1,green,credit card,577,12.5
2,yellow,cash,1412,8.5
3,yellow,credit card,4000,9.5


### **Сводная таблица.**

После `groupby` каждая пара «цвет такси, способ оплаты» занимает отдельную строку. Для сравнения удобнее расположить цвета по строкам, а способы оплаты по столбцам. На их пересечении будет одно число: например, медианная стоимость поездки.

`pivot_table` строит такую **сводную таблицу**: группирует исходные записи по двум направлениям и вычисляет показатель для каждого пересечения. Если вы делали сводные таблицы в Excel, идея та же.

Укажем четыре параметра:

| Параметр | Значение в примере | За что отвечает |
| --- | --- | --- |
| `index` | `"color"` | Какие категории расположить по строкам |
| `columns` | `"payment"` | Какие категории расположить по столбцам |
| `values` | `"fare"` | Из какого столбца брать числа |
| `aggfunc` | `"median"` | Как объединить числа внутри каждой группы |

Например, ячейка строки `yellow` и столбца `card` содержит медиану `fare` всех поездок жёлтого такси с оплатой картой. Если в группе стоимости 10, 20 и 60, в ячейке будет 20. Без явного `aggfunc` используется среднее (`mean`), поэтому выбираем операцию осознанно.

In [14]:
with_payment.pivot_table(
    index="color", columns="payment", values="fare", aggfunc="median"
)

payment,cash,credit card
color,,
green,8.0,12.5
yellow,8.5,9.5


Получили те же медианы, что можно вычислить через `groupby(["color", "payment"])["fare"].median()`, но в форме, удобной для сравнения строк и столбцов.

В одном пересечении может быть много поездок: `aggfunc` превращает их стоимости в одно число. Если для сочетания категорий нет данных, в сводной таблице может появиться пропуск. Это не нулевая стоимость, поэтому автоматически заменять его нулём не стоит.

### **Смысл значений.**

Посмотрим на медианные чаевые по способу оплаты. Цифры легко истолковать неверно, если не знать, что именно записывали в столбец `tip`.

In [15]:
with_payment.groupby("payment")["tip"].median()

payment
cash           0.0
credit card    2.2
Name: tip, dtype: float64

Ноль у наличных **не означает**, что пассажиры никогда не оставляли чаевых. В словарях полей [жёлтых](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) и [зелёных](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf) такси сказано: поле чаевых автоматически заполняется для оплаты картой, а наличные чаевые туда не входят. Сначала выясняем смысл данных, потом делаем вывод.

---

## **Регулярные выражения.**

**Что делать, если известен вид записи, но не её точное значение?** Для слова `Airport` хватало обычного поиска.

### **Почему «регулярные»?**

Название пришло из математики. **Формальный язык** – множество строк, описанное правилами. Например, множество номеров из двух латинских букв, дефиса и трёх цифр.

**Регулярными** называют языки, которые можно описать сочетанием последовательности, выбора и повторения. Регулярное выражение записывает такие правила в компактной форме. Поэтому «регулярное» здесь – название математического класса, а не «часто используемое» или «обязательно повторяющееся».

Практический модуль `Python` `re` умеет больше классической модели. Начнём с простых правил, затем будем их сочетать.

Теперь представим текстовую квитанцию. Ниже вымышленные строки для обучения; это не дополнительные поля исходного `CSV`.

In [16]:
receipt = "Поездка AB-204; дата 15.03.2019 08:05; оплата card"
"AB-204" in receipt

True

Этот поиск сработал, потому что мы уже знали номер. А если номер меняется, но устроен одинаково: две заглавные латинские буквы, дефис и три цифры?

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>Регулярное выражение:</strong><br>Шаблон, описывающий допустимое устройство фрагмента текста. По нему можно найти фрагмент, проверить всю строку или заменить совпадения.
</div>

Для работы с такими шаблонами в стандартной библиотеке `Python` есть модуль `re`. Начнём с буквального текста, чтобы сначала разобраться с результатом поиска.

In [17]:
import re

match = re.search("AB-204", receipt)
match

<re.Match object; span=(8, 14), match='AB-204'>

`re.search(шаблон, текст)` ищет первое совпадение в любом месте строки. Если оно есть, возвращается объект `Match`: он хранит найденный фрагмент и его положение. Метод `group()` возвращает весь совпавший текст.

In [18]:
match.group()

'AB-204'

Метод `span()` возвращает начало и конец совпадения. Конец, как в срезе `Python`, не входит во фрагмент.

In [19]:
start, stop = match.span()
start, stop, receipt[start:stop]

(8, 14, 'AB-204')

Если совпадения нет, результатом будет `None`. У него нет метода `group()`: сначала проверяем результат поиска.

In [20]:
missing_match = re.search("ZZ-999", receipt)
if missing_match is None:
    print("Номер не найден")
else:
    print(missing_match.group())

Номер не найден


### **Последовательность символов.**

В шаблоне обычная буква означает саму себя. Такие элементы называют **литералами**. Записанные подряд литералы должны встретиться подряд и в том же порядке: `AB` ищет именно `AB`, а не любую из этих букв.

Кавычки ограничивают строку `Python` и не входят в шаблон. В вызове `re.search("AB", receipt)` первый аргумент описывает искомое, второй содержит текст, в котором ищем.

In [21]:
print("Найдено AB:", re.search("AB", receipt).group())
print("Результат поиска BA:", re.search("BA", receipt))

Найдено AB: AB
Результат поиска BA: None


### **Классы символов.**

Теперь разрешим несколько вариантов **в одной позиции**. Запись `[ABC]` означает один символ: `A`, `B` или `C`. Скобки не означают строку `ABC` и не переставляют буквы.

Поиск ниже вернёт только первую подходящую букву. Чтобы описать несколько соседних позиций, нужно написать несколько элементов.

In [22]:
re.search("[ABC]", "номер BCA").group()

'B'

Диапазон `[A-Z]` задаёт заглавную латинскую букву, `[0-9]` – ASCII-цифру. Внутри одних скобок можно объединить варианты: `[A-Z0-9]` разрешает букву **или** цифру, но по-прежнему занимает одну позицию.

Дефис между концами диапазона задаёт диапазон. В конце класса он буквальный: `[A-Z0-9-]` разрешает также дефис. Вне скобок обычный дефис тоже обозначает сам себя.

Соберём номер: две позиции для букв, буквальный дефис, три позиции для цифр.

In [23]:
ticket_pattern = "[A-Z][A-Z]-[0-9][0-9][0-9]"
re.search(ticket_pattern, receipt).group()

'AB-204'

### **Повторения.**

Повторы можно записать короче. **Квантификатор** задаёт число повторений непосредственно предыдущего элемента: `{2}` – ровно два, `{3}` – ровно три. Поэтому `[A-Z]{2}` означает две буквы из диапазона. Буквы могут быть разными: повторяется правило выбора, а не найденный символ.

In [24]:
ticket_pattern = "[A-Z]{2}-[0-9]{3}"
re.search(ticket_pattern, receipt).group()

'AB-204'

Другие часто встречающиеся квантификаторы:

| Запись | Сколько повторений предыдущего элемента |
| --- | --- |
| `?` | Ноль или одно |
| `*` | Ноль или больше |
| `+` | Одно или больше |
| `{m,n}` | От `m` до `n` включительно |

Если длина числовой части может быть разной, вместо `{3}` поставим `+`.

In [25]:
re.search("[A-Z]{2}-[0-9]+", "Поездка CD-12045").group()

'CD-12045'

### **Поиск и проверка строки.**

Поиск может найти корректный фрагмент внутри некорректного номера. Например, внутри `AB-2049` есть `AB-204`. Это подходит для поиска фрагмента, но не для проверки номера целиком.

In [26]:
re.search(ticket_pattern, "AB-2049").group()

'AB-204'

`re.fullmatch` требует соответствия всей строки. Проверим два номера и подпишем результат словами.

Объект `Match` в условии считается истинным, `None` – ложным. Поэтому выражение `"подходит" if match_result else "не подходит"` даёт понятную подпись. Проверку `is None` из первого примера можно по-прежнему использовать в обычном коде.

In [27]:
for number in ["AB-204", "AB-2049"]:
    match_result = re.fullmatch(ticket_pattern, number)
    print(number, "подходит" if match_result else "не подходит")

AB-204 подходит
AB-2049 не подходит


### **Необязательные части.**

Разберём остальные повторения на коротких строках. `?` разрешает ноль или одно вхождение предыдущего элемента. В `AB-?204` необязателен только дефис: обе записи `AB204` и `AB-204` подходят.

В выводе используем `repr(text)`, чтобы были видны кавычки, пустая строка и управляющие символы.

In [28]:
for text in ["AB204", "AB-204", "AB--204"]:
    print(repr(text), "подходит" if re.fullmatch("AB-?204", text) else "не подходит")

'AB204' подходит
'AB-204' подходит
'AB--204' не подходит


`*` допускает даже пустую последовательность, а `+` требует хотя бы один символ. Поэтому `[0-9]*` подходит пустой строке. Если поле обязательно должно содержать число, это важное различие.

In [29]:
for text in ["", "7", "204"]:
    star_result = "подходит" if re.fullmatch("[0-9]*", text) else "не подходит"
    plus_result = "подходит" if re.fullmatch("[0-9]+", text) else "не подходит"
    print(repr(text), "для *:", star_result, "; для +:", plus_result)

'' для *: подходит ; для +: не подходит
'7' для *: подходит ; для +: подходит
'204' для *: подходит ; для +: подходит


Для ограниченной длины есть `{m,n}`. Проверим поле из двух, трёх или четырёх цифр. По-прежнему используем `fullmatch`, чтобы лишние цифры не остались за пределами совпадения.

In [30]:
for text in ["7", "12", "1234", "12345"]:
    print(text, "подходит" if re.fullmatch("[0-9]{2,4}", text) else "не подходит")

7 не подходит
12 подходит
1234 подходит
12345 не подходит


### **Строка `Python` и шаблон.**

Буква `r` перед кавычками говорит **`Python`: «сохрани обратные косые черты как написано»**. Это удобно, потому что регулярное выражение использует их для собственных команд.

Посмотрим сначала без регулярки. В обычной строке `"\n"` `Python` превращает два написанных знака в один символ перевода строки. В `r"\n"` остаются два отдельных символа: обратная косая черта и буква `n`.

Напечатаем raw-строку и сравним длины.

In [31]:
print(r"\n")  # На экране будут обратная косая черта и буква n
print(len("\n"), len(r"\n"))  # 1 и 2

\n
1 2


`r"..."` создаёт обычную строку типа `str`, а не особый объект регулярного выражения. Без обратных косых черт разницы нет: `r"abc"` и `"abc"` равны. Одинарные и двойные кавычки здесь тоже равноправны: `r'abc'` и `r"abc"`.

Важно разделять две задачи: `Python` создаёт строку, а `re` читает её содержимое как шаблон. Префикс `r` влияет только на первый шаг. Например, переданное через `r"\n"` обозначение движок `re` всё равно понимает как перевод строки.

### **Экранирование.**

В регулярном выражении `.` означает любой символ, кроме перевода строки при стандартных настройках. Для буквальной точки нужен шаблон `\.`. Обратная косая черта убирает специальный смысл точки: это **экранирование**.

В raw-строке `r"\."` оба символа доходят до `re`. Сравним два шаблона на одной записи.

In [32]:
for pattern in [r"15.03", r"15\.03"]:
    print(pattern, ":", "подходит" if re.fullmatch(pattern, "15x03") else "не подходит")

15.03 : подходит
15\.03 : не подходит


Теперь видно, зачем это удобно. Для поиска буквальной точки движок должен получить два символа: `\.`. Их можно передать двумя способами:

| В коде `Python` | Что получает `re` |
| --- | --- |
| `r"\."` | Обратную косую черту и точку |
| `"\\."` | Те же два символа |

Без `r` обратную косую черту приходится удваивать для `Python`. С `r` пишем шаблон непосредственно в том виде, который нужен регулярке. Проверим, что результат одинаковый.

In [33]:
raw_dot = r"\."
ordinary_dot = "\\."
print("Шаблоны одинаковы:", raw_dot == ordinary_dot)
print("Raw-строка:", re.search(raw_dot, "15.03").group())
print("Обычная строка:", re.search(ordinary_dot, "15.03").group())

Шаблоны одинаковы: True
Raw-строка: .
Обычная строка: .


Сокращение `\d` обозначает десятичную цифру; для строк Unicode это не только символы `0–9`. В наших квитанциях разрешены именно ASCII-цифры, поэтому продолжаем писать `[0-9]`. Сокращение `\s` обозначает пробельный символ: например, пробел, табуляцию или перевод строки.

Даты сначала будем искать как текст вида «две цифры, точка, две цифры, точка, четыре цифры».

In [34]:
date_pattern = r"[0-9]{2}\.[0-9]{2}\.[0-9]{4}"
re.search(date_pattern, receipt).group()

'15.03.2019'

### **Захватывающие группы.**

Круглые скобки выделяют **захватывающую группу**: её содержимое можно получить отдельно. Обрамим день, месяц и год скобками. `group(0)` означает всё совпадение; группы нумеруются с единицы слева направо.

In [35]:
date_parts_pattern = r"([0-9]{2})\.([0-9]{2})\.([0-9]{4})"
date_match = re.search(date_parts_pattern, receipt)
date_match.group(0), date_match.group(1), date_match.group(2), date_match.group(3)

('15.03.2019', '15', '03', '2019')

Части пока остаются строками. Даже весь фрагмент `15.03.2019` ещё не стал объектом даты. Сначала закончим работу с текстом, затем дадим найденному значению календарный смысл.

### **Поиск всех совпадений и замена.**

`re.findall` возвращает все неперекрывающиеся совпадения слева направо. Для шаблона без захватывающих групп результат – список совпавших строк.

In [36]:
date_note = "Посадка 15.03.2019, высадка 16.03.2019"
re.findall(date_pattern, date_note)

['15.03.2019', '16.03.2019']

Группы меняют форму результата `findall`: при одной группе получаем её содержимое, при нескольких – кортежи их содержимого. Проверим наш шаблон с тремя группами.

In [37]:
re.findall(date_parts_pattern, date_note)

[('15', '03', '2019'), ('16', '03', '2019')]

`re.sub(шаблон, замена, текст)` возвращает строку с заменёнными совпадениями. Сведём любую последовательность пробельных символов к одному пробелу. Здесь `\s+` означает «один или больше пробельных символов».

In [38]:
untidy_note = "Поездка	AB-204;   оплата  card"
re.sub(r"\s+", " ", untidy_note)

'Поездка AB-204; оплата card'

### **Жадность и ленивость.**

Квантификаторы по умолчанию **жадные**: берут как можно больше символов, позволяя остальному шаблону совпасть. Точка означает любой символ, а `.*` – любое их количество. Между кавычками такой шаблон может захватить сразу два поля.

In [39]:
quoted_note = 'city="New York"; payment="card"'
re.findall(r'".*"', quoted_note)

['"New York"; payment="card"']

Знак `?` после квантификатора делает его **ленивым**: `*?`, `+?`, `??`, `{m,n}?`. Он начинает с минимального числа повторений и увеличивает его, если остальная часть шаблона не совпала.

В `.*?` вопросительный знак меняет поведение `*`. Самостоятельный `?` по-прежнему означает «ноль или одно повторение». Сравним извлечение тех же полей.

In [40]:
re.findall(r'".*?"', quoted_note)

['"New York"', '"card"']

Можно точнее описать формат: внутри поля разрешены любые символы, кроме кавычки. В начале класса символов `^` означает отрицание набора: `[^"]` – один символ, который не является кавычкой. Квантификатор `*` повторяет этот элемент. Такой шаблон останавливается у закрывающей кавычки и не захватывает соседнее поле.

In [41]:
re.findall(r'"[^"]*"', quoted_note)

['"New York"', '"card"']

Этот пример предполагает, что внутри поля нет экранированных кавычек. Для настоящего `CSV` или `JSON` используем парсер соответствующего формата: усложнять регулярку до самодельного парсера обычно невыгодно.

## **Разбор текстовых логов.**

**Как найти ошибки приложения среди тысяч сообщений?** Лог – последовательность записей о событиях программы. Если формат текстовый, нам нужны границы полей и варианты допустимых значений. Начнём с уровня сообщения: `INFO` – информационная запись, `WARNING` – предупреждение, `ERROR` – ошибка.

Символ `|` задаёт альтернативу: подходит выражение слева или справа. Это выбор, а не квантификатор повторения. `findall` ниже использует шаблон без групп и возвращает найденные фрагменты.

In [42]:
level_text = "INFO started; WARNING slow; ERROR failed"
re.findall(r"INFO|WARNING|ERROR", level_text)

['INFO', 'WARNING', 'ERROR']

### **Область альтернативы.**

Альтернатива выбирает целую ветку: `cat|dog` означает `cat` или `dog`. Чтобы добавить общую часть, заключаем варианты в скобки. В `file\.(jpg|png)` после `file.` должно идти одно из двух расширений.

Без скобок `file\.jpg|png` означало бы «вся строка `file.jpg` или отдельная строка `png`». Круглые скобки здесь и объединяют варианты, и создают уже знакомую захватывающую группу.

In [43]:
re.fullmatch(r"file\.(jpg|png)", "file.png").group(), re.fullmatch(r"file\.jpg|png", "png").group()

('file.png', 'png')

### **Сокращение \w.**

`\w` обозначает один «словесный» символ: Unicode-букву, цифру или `_`. Это не только латиница. `\w+` найдёт последовательность таких символов; пробел и дефис её прервут.

In [44]:
re.findall(r"\w+", "ERROR_code рейс_204 AB-204")

['ERROR_code', 'рейс_204', 'AB', '204']

Получили `ERROR_code`, `рейс_204`, `AB` и `204`. Подчёркивание входит в найденные фрагменты, дефис разделяет их. Если формат разрешает только латиницу и ASCII-цифры, задаём эти ограничения явно через `[A-Za-z0-9_]`.

### **Граница слова.**

`\b` проверяет границу между символом `\w` и символом вне `\w`, а также край входа рядом с `\w`. Сама граница символов не поглощает. Поэтому она может находиться не только возле пробела, но и возле скобки.

В `\bERROR\b` требуем такую границу и перед `ERROR`, и после него. Записываем шаблон с `r`: иначе `Python` воспримет `"\b"` как управляющий символ возврата на одну позицию.

In [45]:
re.findall(r"\bERROR\b", "ERROR ERRORS ERROR_code (ERROR)")

['ERROR', 'ERROR']

Результат – `['ERROR', 'ERROR']`. Разберём четыре фрагмента:

| Фрагмент | Совпадает? | Почему |
| --- | --- | --- |
| `ERROR` | Да | Слева край входа, справа пробел |
| `ERRORS` | Нет | Между `R` и `S` нет границы: оба входят в `\w` |
| `ERROR_code` | Нет | `_` тоже входит в `\w`, границы после `R` нет |
| `(ERROR)` | Да | Скобки не входят в `\w`; обе границы есть |

Скобки в последнем фрагменте не входят в совпадение: шаблон проверяет позиции рядом с буквами.

### **Начало и конец строки.**

Якоря проверяют **позицию**, не поглощая символ. `^` требует начала входа, `$` – конца либо позиции перед единственным завершающим переводом строки. Без флагов речь идёт обо всём переданном тексте.

Сравним обычный поиск `ERROR` с поиском только в начале.

In [46]:
print("Без якоря:", re.findall(r"ERROR", "INFO ERROR"))
print("С якорем ^:", re.findall(r"^ERROR", "INFO ERROR"))

Без якоря: ['ERROR']
С якорем ^: []


Для строгой проверки всей строки удобнее `fullmatch`. У `$` есть оговорка про завершающий перевод строки, поэтому запись `^...$` не всегда заменяет этот метод.

In [47]:
print("Поиск с $:", re.findall(r"^ERROR$", "ERROR\n"))
print("Полное совпадение:", re.fullmatch(r"ERROR", "ERROR\n"))

Поиск с $: ['ERROR']
Полное совпадение: None


### **Флаги поиска.**

Флаг меняет настройки поиска; передаём его через `flags`.

| Флаг | Что меняет | Пример |
| --- | --- | --- |
| `re.IGNORECASE` | Не учитывает регистр | `re.findall(r"error", "ERROR", flags=re.IGNORECASE)` даёт `['ERROR']` |
| `re.MULTILINE` | `^` и `$` учитывают границы каждой строки | `re.findall(r"^ERROR", "INFO\nERROR", flags=re.MULTILINE)` даёт `['ERROR']` |
| `re.DOTALL` | Точка совпадает также с переводом строки | `re.findall(r"a.b", "a\nb", flags=re.DOTALL)` даёт `['a\nb']` |
| `re.VERBOSE` | Позволяет писать шаблон с комментариями и отступами | `re.fullmatch(r"[0-9]+  # число", "204", flags=re.VERBOSE).group()` даёт `'204'` |

Несколько флагов объединяем через `|`. Найдём `ERROR` в начале любой строки независимо от регистра. Это операция над флагами `Python`, не альтернатива внутри `regex`.

In [48]:
multiline_log = "INFO started\nerror failed\nWARNING slow"
re.findall(r"^ERROR", multiline_log, flags=re.MULTILINE | re.IGNORECASE)

['error']

### **Незахватывающие группы.**

Скобки решают две задачи: объединяют части шаблона и сохраняют совпавший текст как отдельную группу. Иногда нужна только первая задача: выбрать `INFO` или `ERROR`, затем дописать номер, но извлечь всю запись целиком.

`(?:...)` объединяет выражения **без отдельного захвата**. Это полезно, чтобы служебные скобки не меняли результат `findall` и не добавляли номера групп. Сравним:

In [49]:
print("Захватывающая группа:", re.findall(r"(INFO|ERROR)-[0-9]+", "INFO-12 ERROR-34"))
print("Незахватывающая группа:", re.findall(r"(?:INFO|ERROR)-[0-9]+", "INFO-12 ERROR-34"))

Захватывающая группа: ['INFO', 'ERROR']
Незахватывающая группа: ['INFO-12', 'ERROR-34']


Обычные скобки дали только уровни сообщений. С `(?:...)` отдельных захватывающих групп нет, поэтому `findall` вернул записи целиком. Обе версии находят те же участки текста; меняется то, что мы извлекаем.

### **Проверка имени файла.**

Соберём знакомые элементы. Пусть имя состоит из латинских букв, цифр, `_` и дефиса, а расширение – только `jpg`, `png` или `gif`. Это правило нашего учебного сервиса, а не универсальное правило имён файлов.

<img src="../../Other/images/05_regex.png" width="1000" style="max-width: 100%; height: auto;" alt="Шаблон имени изображения: начало строки, допустимые символы имени, точка, одно из трёх расширений, конец строки">

Прочитаем схему по частям:

| Фрагмент | Что проверяет |
| --- | --- |
| `^` | Начало входа |
| `[a-zA-Z0-9_-]+` | Хотя бы один допустимый символ имени; дефис в конце класса буквальный |
| `\.` | Точку перед расширением |
| `(jpg\|png\|gif)` | Одно из трёх расширений |
| `$` | Конец входа с уже разобранной оговоркой про перевод строки |

В `Python` проверим имя через `fullmatch`: якоря тогда не нужны. Расширение отдельно извлекать не будем, поэтому заменим захватывающие скобки на `(?:...)`. Смысл проверки имени сохранится.

In [50]:
image_name_pattern = r"[a-zA-Z0-9_-]+\.(?:jpg|png|gif)"
image_names = [
    "trip-204.png", "a.jpg", ".png", "trip.jpeg",
    "trip.PNG", "trip.png.bak", "trip.png\n",
]
for name in image_names:
    print(repr(name), "подходит" if re.fullmatch(image_name_pattern, name) else "не подходит")

'trip-204.png' подходит
'a.jpg' подходит
'.png' не подходит
'trip.jpeg' не подходит
'trip.PNG' не подходит
'trip.png.bak' не подходит
'trip.png\n' не подходит


Подходят первые два имени. Пустая основа, другое расширение, заглавные буквы расширения и лишний хвост не входят в наше правило. Регистр при необходимости меняется знакомым флагом `re.IGNORECASE`.

Так мы проверили только имя. Расширение `.png` не доказывает, что содержимое файла действительно является изображением.

Группе можно дать имя: `(?P<level>...)`. Тогда `group("level")` читает поле по смыслу, без запоминания номера. Метод `groupdict()` возвращает словарь всех именованных групп.

В нашем небольшом формате номер запроса содержит только заглавные латинские буквы, ASCII-цифры и дефис; уровень отделён одним пробелом. Сначала разберём запись без времени.

In [51]:
service_line = "ERROR request=AB-204"
service_pattern = r"(?P<level>INFO|WARNING|ERROR) request=(?P<request>[A-Z0-9-]+)"
service_match = re.fullmatch(service_pattern, service_line)
service_match.groupdict()

{'level': 'ERROR', 'request': 'AB-204'}

Шаблон растёт, и его становится труднее читать. `re.VERBOSE` позволяет записать его на нескольких строках и добавить комментарии через `#`. Обычные пробелы вне классов символов при этом игнорируются. Для нужного пробела в данных напишем `[ ]`, а для буквального `#` потребовалось бы экранирование или класс символов.

In [52]:
readable_service_pattern = r"""
(?P<level>INFO|WARNING|ERROR)  # Уровень сообщения
[ ]request=                  # Разделитель и имя поля
(?P<request>[A-Z0-9-]+)       # Идентификатор запроса
"""
re.fullmatch(readable_service_pattern, service_line, flags=re.VERBOSE).groupdict()

{'level': 'ERROR', 'request': 'AB-204'}

Читаемый шаблон проще проверять на успешных и неуспешных примерах. Но если сервис уже пишет структурированный `JSON`, извлекайте поля `JSON`-парсером, а не регуляркой: формат и экранирование уже описаны стандартом. `Regex` пригодится для старого текстового лога или извлечения фрагмента из поля сообщения.

### **Проверка окружения.**

Иногда нужно найти фрагмент при условии, что рядом есть определённый текст, но не включать этот текст в результат. Например, извлечь число только перед обозначением рублей.

**Опережающая проверка**, или lookahead, записывается как `(?=...)`. Она проверяет текст справа от текущей позиции и сама ничего не добавляет к совпадению. Сравним с обычным шаблоном, который захватывает и число, и валюту.

In [53]:
price_note = "Цена 500 руб.; доставка 20 USD"
re.findall(r"[0-9]+ руб\.", price_note), re.findall(r"[0-9]+(?= руб\.)", price_note)

(['500 руб.'], ['500'])

Во втором результате осталось только `500`: условие про ` руб.` проверено, но эти символы не вошли в совпадение.

**Ретроспективная проверка**, или lookbehind, смотрит влево: `(?<=...)`. Найдём число, непосредственно перед которым написано `Цена `. Само слово в результат не включается.

In [54]:
re.findall(r"(?<=Цена )[0-9]+", price_note)

['500']

Четыре варианта проверки окружения:

| Конструкция | Условие |
| --- | --- |
| `(?=...)` | Справа должно быть совпадение с `...` |
| `(?!...)` | Справа не должно быть совпадения с `...` |
| `(?<=...)` | Слева должно быть совпадение с `...` |
| `(?<!...)` | Слева не должно быть совпадения с `...` |

Выберем целые числовые фрагменты, после которых нет ` руб.`. Границы `\b` не позволяют взять только часть числа `500`, чтобы обойти проверку валюты.

In [55]:
re.findall(r"\b[0-9]+\b(?! руб\.)", price_note)

['20']

Получили `20`. Проверки окружения, как знакомые якоря, задают условие для позиции. Это не захватывающие группы; найденный текст по-прежнему задаёт `[0-9]+`.

В стандартном `re` lookbehind должен иметь фиксированную длину. `(?<=Цена )` проверяет ровно пять символов и допустим. `(?<=[0-9]+)` недопустим: число символов заранее не определено. Если задача требует такого контекста, часто проще включить его в обычный шаблон и извлечь нужную часть захватывающей группой.

Теперь разберём, как движок выполняет шаблон и выбирает совпадение.

---

## **Устройство `regex`-движка.**

**Мы описали шаблон. Кто выполняет его и почему один поиск бывает дороже другого?** В CPython модуль `re` разбирает шаблон и компилирует его во внутренние инструкции. Эти инструкции исполняет движок сопоставления, написанный на C. Это не генерация машинного кода для нашего шаблона.

Для понимания удобно мысленно отслеживать две позиции: текущий символ текста и текущую часть шаблона. Литерал проверяет конкретный символ, класс – принадлежность набору, квантификатор – повторение. `search` ищет подходящее начало, `fullmatch` требует покрытия всей строки. Реальный движок умеет пропускать заведомо неподходящие позиции; это модель для рассуждения, а не утверждение о точном числе внутренних шагов.

`re.compile` возвращает объект `Pattern`, который можно сохранить и использовать повторно. Он ещё не ищет совпадение: поиск начинается при вызове его методов.

In [56]:
compiled_ticket = re.compile(ticket_pattern)
compiled_ticket.fullmatch("AB-204").group()

'AB-204'

Функции `re.search` и другие тоже используют компиляцию и кэш последних шаблонов. Поэтому явный `compile` полезен для организации кода и повторного использования, но не гарантирует ускорения каждого вызова.

### **Жадность и возврат.**

Возьмём маленький шаблон `a+a` и текст `aaa`. Первый `a+` сначала забирает все три буквы. Для последнего `a` ничего не осталось. Движок возвращает одну букву из повторения и пробует остаток шаблона снова: теперь совпадение есть. Такой перебор с возвращением к ранее сделанному выбору называется **backtracking**.

In [57]:
re.fullmatch(r"a+a", "aaa").group()

'aaa'

| Шаг учебной трассировки | Что забрал `a+` | Что осталось для последнего `a` |
| --- | --- | --- |
| Жадный выбор | `aaa` | Ничего: продолжение не совпало |
| Возврат на один символ | `aa` | `a`: весь шаблон совпал |

Теперь понятно, почему жадность означает «максимально много при успешном совпадении всего шаблона». Возврат позволяет пересмотреть первый выбор.

### **Стоимость поиска.**

В шаблоне `(a+)+` одна и та же цепочка букв допускает много разбиений между вложенными повторениями. Если в конце строки встретится `!`, движок может перебрать множество вариантов, прежде чем сообщит об отсутствии полного совпадения. Количество вариантов быстро растёт с длиной строки. Это называют катастрофическим backtracking.

Проверим только короткий пример. Оба шаблона для `fullmatch` описывают одну или больше букв `a`, но у простого нет вложенной неоднозначности.

In [58]:
short_failure = "a" * 8 + "!"
for pattern in [r"(a+)+", r"a+"]:
    print(pattern, ":", "совпало" if re.fullmatch(pattern, short_failure) else "не совпало")

(a+)+ : не совпало
a+ : не совпало


Ленивость сама по себе не устраняет перебор. Для больших входов полезнее убирать неоднозначные повторы, задавать допустимые символы и проверять не только успешные примеры, но и строки, которые почти подходят. У стандартного `re` нет параметра тайм-аута поиска; длинный эксперимент с таким шаблоном может надолго занять ядро ноутбука.

### **Экранирование и Unicode.**

Если фрагмент шаблона приходит из обычного текста, его специальные символы надо экранировать. `re.escape` делает это автоматически. Найдём буквальное `a+b`, не превращая `+` в квантификатор.

In [59]:
literal_label = "a+b"
re.search(re.escape(literal_label), "поле a+b заполнено").group()

'a+b'

Вторая тонкость – Unicode. Шаблон `\d` принимает не только ASCII-цифры. Поэтому внешне похожие правила проверки могут описывать разные множества строк.

In [60]:
for pattern in [r"\d+", r"[0-9]+"]:
    print(pattern, ":", "подходит" if re.fullmatch(pattern, "١٢٣") else "не подходит")

\d+ : подходит
[0-9]+ : не подходит


И ещё один выбор инструмента: если нужен только буквальный фрагмент, достаточно `in`, `str.find` или `.str.contains(..., regex=False)`. Регулярки оправданы там, где действительно есть переменная структура. Теперь применим знакомый шаблон к столбцу.

---

## **Регулярные выражения в `Pandas`.**

Для столбца используем знакомый доступ `.str`. Шаблон остаётся тем же; метод применяется к каждому значению.

| Задача | Метод |
| --- | --- |
| Отметить строки с совпадением | `.str.contains(pattern, regex=True, na=False)` |
| Извлечь захватывающую группу | `.str.extract(pattern, expand=False)` |
| Заменить совпадения | `.str.replace(pattern, replacement, regex=True)` |

Возьмём четыре записи. Вторая содержит невозможное 31 февраля: оно поможет отличить проверку текста от проверки календаря.

In [61]:
receipts = pd.Series([
    "  Поездка AB-204; дата 15.03.2019 08:05  ",
    "Поездка CD-120; дата 31.02.2019 09:10",
    "Поездка EF-300; дата неизвестна",
    None,
], name="receipt", dtype="string")
receipts

0      Поездка AB-204; дата 15.03.2019 08:05  
1        Поездка CD-120; дата 31.02.2019 09:10
2              Поездка EF-300; дата неизвестна
3                                         <NA>
Name: receipt, dtype: string

Найдём записи с датой и извлечём её текст. `na=False` даёт `False` для пропуска при поиске; `extract` оставляет пропуск, если совпадения нет. Скобки вокруг `date_pattern` задают одну извлекаемую группу.

In [62]:
date_texts = receipts.str.extract(f"({date_pattern})", expand=False)
pd.DataFrame({
    "есть_дата": receipts.str.contains(date_pattern, regex=True, na=False),
    "текст_даты": date_texts,
})

,есть_дата,текст_даты
0,True,15.03.2019
1,True,31.02.2019
2,False,<NA>
3,False,<NA>


Заменим последовательности пробельных символов одним пробелом. `strip()` уберёт оставшиеся пробелы по краям; пропуски сохранятся.

In [63]:
receipts.str.replace(r"\s+", " ", regex=True).str.strip()

0    Поездка AB-204; дата 15.03.2019 08:05
1    Поездка CD-120; дата 31.02.2019 09:10
2          Поездка EF-300; дата неизвестна
3                                     <NA>
Name: receipt, dtype: string

Шаблон извлёк и `15.03.2019`, и `31.02.2019`: обе строки имеют нужное устройство. Чтобы проверить календарный смысл и посчитать длительность, перейдём к типам даты и времени.

---

## **Представление времени.**

Сервер записал «12:00», а платёжная система – «12:05». Прошло пять минут? Пока неизвестно: нужны дата и сведения о часах каждого источника. Разберём, что хранить, чтобы правильно сравнивать события.

### **Момент, дата и длительность.**

| Что описываем | На примере поездки |
| --- | --- |
| Момент | Когда началась конкретная поездка |
| Местные дата и часы | Что показывали часы в месте отправления |
| Длительность | Сколько времени прошло до прибытия |

Начало и конец – два момента. Их можно записать разными местными часами. Длительность 15 минут сама по себе не говорит, когда поездка состоялась.

### **Календарь и единицы.**

Календарь организует дни в месяцы и годы: месяцы имеют разную длину, в високосные годы есть 29 февраля. Поэтому «через месяц» не всегда означает «через 30 дней».

Для длительностей нужна воспроизводимая единица. Современная секунда определяется через частоту перехода в атоме цезия-133; вращение Земли недостаточно равномерно для такого эталона. Для нашего кода важен практический вывод: единица времени и точность конкретных часов – разные вещи.

### **Единицы времени.**

В обычном счёте длительностей минута – 60 секунд, час – 60 минут. Для запросов к серверу секунды часто слишком крупны: два события могут произойти внутри одной секунды. Тогда используем её доли.

| Единица | Доля секунды | Сколько в одной секунде |
| --- | --- | --- |
| Миллисекунда (`ms`) | 1 / 1 000 | 1 000 |
| Микросекунда (`μs`, в коде `us`) | 1 / 1 000 000 | 1 000 000 |
| Наносекунда (`ns`) | 1 / 1 000 000 000 | 1 000 000 000 |

Между соседними строками таблицы множитель 1000. Например, 0,123 секунды – это 123 миллисекунды, 123 000 микросекунд или 123 000 000 наносекунд. Длительность одна, меняется единица записи.

**Разрешение и точность.** Если формат сохраняет наносекунды, он различает очень мелкие отсчёты. Но часы сервера при этом могут отставать на две секунды. Число знаков в метке не доказывает точность измерения. Позже проверим также потерю разрядов при преобразовании типов.

### **`UTC` и смещение.**

Для расписаний удобно местное время, а для сопоставления событий из разных городов нужна общая шкала. **`UTC`** – всемирное координированное время; у него нет сезонного перевода часов.

**Смещение** показывает разность местных часов и `UTC`. `+03:00` означает «на три часа впереди `UTC`». Один момент оплаты можно записать так:

| Представление | Дата и время |
| --- | --- |
| `UTC` | 15 января 2026, 09:00 |
| Со смещением `+03:00` | 15 января 2026, 12:00 |
| Со смещением `-05:00` | 15 января 2026, 04:00 |

Событие одно. Для местного представления прибавляем смещение к `UTC`; для обратного перевода вычитаем. При переходе через полночь меняется и дата.

### **Часовой пояс.**

Смещение относится к конкретному моменту. **Региональный пояс** задаёт правила выбора смещения для разных дат. Например, в Нью-Йорке зимой и летом 2026 года действуют `-05:00` и `-04:00`.

Правила устанавливают власти; они могут меняться. База **`IANA`** хранит их историю под именами вроде `America/New_York`. Поэтому для расписания «каждый день в 09:00» сохраняют регион, а не одно фиксированное смещение. [База `IANA`](https://data.iana.org/time-zones/tz-link.html).

Карта иллюстрирует смещения относительно `UTC`. Её дата актуальности неизвестна; действующие правила для региона и даты берём из базы `IANA`.

<a href="../../Other/images/05_time_zones.png"><img src="../../Other/images/05_time_zones.png" width="1000" style="max-width: 100%; height: auto;" alt="Карта смещений относительно UTC"></a>

### **Перевод часов.**

При сезонном переводе меняются показания местных часов. Рассмотрим два случая на правилах Нью-Йорка.

| Случай | Показания до перевода | Следующее показание | Что происходит с `02:30` или `01:30` |
| --- | --- | --- | --- |
| Весной | `01:59:59` | `03:00:00` | `02:30` в этот день не существует |
| Осенью | `01:59:59` | `01:00:00` | `01:30` встречается дважды |

Повторный час создаёт **неоднозначное** местное время. Даже зная дату и регион, нужно уточнить, какое из двух повторений имеется в виду.

Между соседними местными полуночами при переводе на час может пройти 23 или 25 часов. Поэтому «через 24 часа» и «завтра в то же время» – разные условия. Для длительности сравниваем моменты на общей шкале; для расписания применяем местный календарь.

### **`Unix Epoch` и `timestamp`.**

Чтобы записать момент числом, выберем начало отсчёта и единицу. **`Unix Epoch`** – 1 января 1970 года, 00:00:00 `UTC`. **Unix `timestamp`** – числовая метка относительно этой эпохи, обычно в секундах.

`0` соответствует эпохе, `60` – минуте после неё, `-60` – минуте до неё. Выбор 1970 года – соглашение. Один момент даёт одну Unix-метку независимо от местного представления.

В API передают также миллисекунды и наносекунды. Число без единицы неоднозначно: `1000` секунд и `1000` миллисекунд задают разные моменты. Единицу берём из описания поля.

**Ограничение модели.** Для согласования `UTC` с вращением Земли вводили дополнительные секунды, записываемые как `23:59:60`. Это не сезонный перевод часов. Unix-время в модели `POSIX` отдельно их не считает, а `Python` `datetime` не представляет секунду `60`. Для обычной аналитики эта модель удобна; для физических измерений через такие поправки нужно уточнять шкалу времени.

Теперь выразим знакомые понятия в `Python`: создадим календарные значения, зададим смещение, получим Unix-метку и применим правила региона.

---

## **Время в `Python`.**

Для разобранных значений стандартный модуль `datetime` даёт отдельные типы. Выберем тип по смыслу данных.

| Тип | Что хранит |
| --- | --- |
| `date` | Календарную дату: год, месяц, день |
| `time` | Время суток: часы, минуты, секунды, микросекунды; может содержать информацию о поясе |
| `datetime` | Дату и время суток; может содержать информацию о поясе |
| `timedelta` | Длительность, например 15 минут или 2 дня |

`Python` использует продолженный григорианский календарь: знакомые правила распространяются на весь поддерживаемый диапазон лет, без моделирования исторического перехода стран на этот календарь.

Модуль и один из его классов называются одинаково. Ниже импортируем классы из модуля; имя `datetime` в коде будет обозначать класс.

In [64]:
from datetime import date, time, datetime, timedelta

Начнём с календарной даты. В конструкторе порядок чисел – год, месяц, день. Указывать формат строки не нужно: передаём числа.

In [65]:
trip_day = date(2019, 3, 15)
trip_day, trip_day.year, trip_day.month, trip_day.day

(datetime.date(2019, 3, 15), 2019, 3, 15)

Объект `time` хранит время суток без конкретного календарного дня. Два одинаковых времени суток могут относиться к разным датам.

In [66]:
departure_clock = time(23, 55)
departure_clock, departure_clock.hour, departure_clock.minute

(datetime.time(23, 55), 23, 55)

У `datetime` есть обе части. Создадим начало и конец поездки, которая пересекает полночь. Здесь время записано без часового пояса; пока сравниваем такие местные показания часов в рамках примера.

In [67]:
departure = datetime(2019, 3, 15, 23, 55)
arrival = datetime(2019, 3, 16, 0, 10)
departure, arrival

(datetime.datetime(2019, 3, 15, 23, 55), datetime.datetime(2019, 3, 16, 0, 10))

Методы `.date()` и `.time()` возвращают части этого значения отдельными объектами. Если оставить только часы, мы потеряем информацию о смене дня.

In [68]:
departure.date(), departure.time()

(datetime.date(2019, 3, 15), datetime.time(23, 55))

Разность двух значений `datetime` – объект `timedelta`. Между записанными календарными показаниями прошло 15 минут. В этом простом примере предполагаем, что часы не переводились и их смещение не менялось, поэтому разность совпадает с длительностью поездки.

In [69]:
elapsed = arrival - departure
elapsed

datetime.timedelta(seconds=900)

Продолжительность можно создать и самостоятельно, а затем прибавить к дате со временем. У `timedelta` нет единицы «месяц»: месяцы имеют разное число дней.

In [70]:
departure + timedelta(minutes=15)

datetime.datetime(2019, 3, 16, 0, 10)

### **Компоненты длительности.**

`timedelta` хранит нормализованные компоненты: дни, секунды внутри дня и микросекунды. Атрибут `.seconds` находится в диапазоне от `0` до `86399`; это не число секунд внутри минуты и не полная длительность.

In [71]:
long_elapsed = timedelta(days=1, minutes=2, seconds=3)
long_elapsed.days, long_elapsed.seconds, long_elapsed.microseconds

(1, 123, 0)

Метод `total_seconds()` переводит всю длительность в секунды. Чтобы получить минуты числом, делим результат на 60.

In [72]:
long_elapsed.total_seconds(), long_elapsed.total_seconds() / 60

(86523.0, 1442.05)

### **Строка и дата.**

До этого мы передавали конструктору числа. Теперь прочитаем дату из квитанции. `strptime` разбирает строку по формату: `%d` – день, `%m` – месяц, `%Y` – четырёхзначный год. Точки в формате обозначают буквальные точки; это формат даты, а не регулярное выражение.

In [73]:
receipt_day = datetime.strptime(date_texts.iloc[0], "%d.%m.%Y")
receipt_day

datetime.datetime(2019, 3, 15, 0, 0)

Если часы не указаны, в полученном `datetime` будет полночь. `.date()` оставит только календарную дату. Невозможное число месяца вызывает `ValueError`: перехватим его, чтобы учебный пример не прерывал выполнение ноутбука.

In [74]:
try:
    datetime.strptime("31.02.2019", "%d.%m.%Y")
except ValueError as error:
    print(type(error).__name__, str(error))

ValueError day 31 must be in range 1..28 for month 2 in year 2019


Шаблон проверил текст, а `strptime` проверил календарный смысл. Обратная операция – `strftime`: из объекта времени она получает строку в указанном формате. `%H` задаёт часы в 24-часовом виде, `%M` – минуты, `%S` – секунды. Строчная `%m` и заглавная `%M` означают разные вещи.

In [75]:
departure.strftime("%d.%m.%Y %H:%M:%S")

'15.03.2019 23:55:00'

В исходном `CSV` поездок даты записаны в порядке год–месяц–день, затем часы. Проверим, что после `read_csv` это пока текст, а не временные значения.

In [76]:
taxi[["pickup", "dropoff"]].dtypes

pickup     str
dropoff    str
dtype: object

`datetime.fromisoformat` разбирает поддерживаемые ISO-представления даты со временем без отдельной строки формата. Такой порядок используется и в нашем `CSV`; пробел между датой и временем допустим.

In [77]:
pickup_py = datetime.fromisoformat(taxi.loc[0, "pickup"])
dropoff_py = datetime.fromisoformat(taxi.loc[0, "dropoff"])
pickup_py, dropoff_py

(datetime.datetime(2019, 3, 23, 20, 21, 9),
 datetime.datetime(2019, 3, 23, 20, 27, 24))

Мы снова получили два отдельных объекта. Вычтем их и переведём длительность в минуты тем способом, который уже проверили на примере больше суток.

In [78]:
(dropoff_py - pickup_py).total_seconds() / 60

6.25

## **Смещение в `Python`.**

Вернёмся к одному событию, записанному как 09:00 `UTC` и 12:00 со смещением `+03:00`. Теперь создадим обе записи в `Python` и сравним их.

`timezone` из модуля `datetime` описывает фиксированное смещение. Константа `UTC` задаёт нулевое смещение. Параметр `tzinfo` прикрепляет сведения о поясе к объекту; он не меняет переданные числа часов.

In [79]:
from datetime import UTC, timezone

fixed_plus_three = timezone(timedelta(hours=3))
utc_event = datetime(2026, 1, 15, 9, 0, tzinfo=UTC)
local_event = datetime(2026, 1, 15, 12, 0, tzinfo=fixed_plus_three)
utc_event == local_event

True

Часы разные, но события совпадают. Такие значения называют **aware**: по ним можно определить момент относительно `UTC`. Значение без определённого смещения называют **naive**. Точный критерий aware для `datetime`: `tzinfo` не равен `None` и `utcoffset()` возвращает смещение, а не `None`. Метод `utcoffset()` показывает разность местных часов и `UTC`.

In [80]:
local_event.utcoffset(), departure.tzinfo is None

(datetime.timedelta(seconds=10800), True)

### **Преобразование `timestamp`.**

Теперь получим числовую координату момента относительно знакомой `Unix Epoch`. Метод `.timestamp()` переводит aware-значение в Unix-секунды. Проверим ноль эпохи и две записи одного события.

In [81]:
epoch = datetime(1970, 1, 1, tzinfo=UTC)
epoch.timestamp(), utc_event.timestamp(), local_event.timestamp()

(0.0, 1768467600.0, 1768467600.0)

Один момент дал одно число независимо от представления часами. Обратную операцию выполняет `datetime.fromtimestamp`. Явно передадим `tz=UTC`, чтобы результат не зависел от настроек компьютера.

In [82]:
event_seconds = utc_event.timestamp()
datetime.fromtimestamp(event_seconds, tz=UTC)

datetime.datetime(2026, 1, 15, 9, 0, tzinfo=datetime.timezone.utc)

Почему `datetime(1970, 1, 1).timestamp()` не обязан дать ноль? Здесь нет пояса. Для naive-объекта `.timestamp()` предполагает местный пояс компьютера; `fromtimestamp()` без `tz` тоже использует местное время и возвращает naive-объект. Поэтому воспроизводимые примеры выше задают пояс явно.

Для текущего момента используйте `datetime.now(UTC)`. В учебных расчётах оставляем фиксированную дату, чтобы результат не менялся между запусками.

### **Миллисекунды в `Python`.**

`fromtimestamp` принимает секунды. Если источник передал целые миллисекунды, переводим их в секунды делением на 1000. Единицу для этого примера мы знаем заранее.

In [83]:
event_milliseconds = 1_700_000_000_123
datetime.fromtimestamp(event_milliseconds / 1000, tz=UTC)

datetime.datetime(2023, 11, 14, 22, 13, 20, 123000, tzinfo=datetime.timezone.utc)

Стандартный `datetime` имеет микросекундное разрешение. `.timestamp()` возвращает `float`; не всякая десятичная дробь представима им точно. Если исходник даёт целое число миллисекунд или наносекунд, сохраняем исходное число и единицу, чтобы не потерять младшие разряды при ненужном переводе через `float`.

Позже в `Pandas` сравним два события, разделённые наносекундой, и увидим эту потерю на конкретных значениях.

---

## **Региональный пояс в `Python`.**

Фиксированное смещение мы создали через `timezone`. Для правил региона используем `ZoneInfo` из стандартного модуля `zoneinfo`, доступного с `Python` 3.9.

Сначала получим объект с правилами Нью-Йорка. Он описывает пояс, а не дату и не текущее время.

In [84]:
from zoneinfo import ZoneInfo

new_york = ZoneInfo("America/New_York")
new_york

zoneinfo.ZoneInfo(key='America/New_York')

Передадим этот объект в `tzinfo` двух календарных значений: зимнего и летнего полудня. Для этих дат местное время однозначно. Затем `utcoffset()` вычислит смещение каждого значения по правилам региона.

In [85]:
winter_noon = datetime(2026, 1, 15, 12, tzinfo=new_york)
summer_noon = datetime(2026, 7, 15, 12, tzinfo=new_york)
winter_noon.utcoffset(), summer_noon.utcoffset()

(datetime.timedelta(days=-1, seconds=68400),
 datetime.timedelta(days=-1, seconds=72000))

`Python` нормализует отрицательный `timedelta` так, чтобы компонент секунд оставался неотрицательным. Поэтому `days=-1, seconds=68400` означает `-86400 + 68400 = -18000` секунд, то есть **-5 часов**. Второй результат, `days=-1, seconds=72000`, означает **-4 часа**. Это те же зимнее и летнее смещения, записанные через компоненты длительности.

### **Данные часовых поясов.**

`zoneinfo` читает системную базу `IANA`, а при её отсутствии использует пакет `tzdata`. Если нет обоих источников, возникает `ZoneInfoNotFoundError`. На Windows часто требуется установить `tzdata`:

```bash
python -m pip install tzdata
```

Правила могут меняться, поэтому версия базы важна для воспроизводимости расчётов. `ZoneInfo` не загружает свежие правила из интернета при каждом вызове. [Документация zoneinfo](https://docs.python.org/3.14/library/zoneinfo.html).

### **Назначение и преобразование пояса.**

Метод `.astimezone()` сохраняет момент, но вычисляет местные часы в другом поясе. Переведём наш `utc_event` в Нью-Йорк и проверим числовую метку.

In [86]:
ny_event = utc_event.astimezone(new_york)
ny_event, ny_event.timestamp() == utc_event.timestamp()

(datetime.datetime(2026, 1, 15, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 True)

А `.replace(tzinfo=...)` сохраняет написанные часы и заменяет сведения о поясе. Для уже известного момента это меняет смысл записи. Сравним: «09:00 `UTC` показать в Нью-Йорке» и «считать 09:00 местными часами Нью-Йорка».

In [87]:
reinterpreted_event = utc_event.replace(tzinfo=new_york)
ny_event, reinterpreted_event, reinterpreted_event.timestamp() - utc_event.timestamp()

(datetime.datetime(2026, 1, 15, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 datetime.datetime(2026, 1, 15, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 18000.0)

Назначать пояс naive-значению можно лишь зная, какие часы записал источник. Более того, назначение через `replace` само по себе не проверяет существование местного времени при переводе часов. Эту границу разберём ниже.

Вычитать aware- и naive-значения нельзя: у одного определён момент, а у другого не хватает информации. `Python` не угадывает недостающий пояс.

In [88]:
try:
    utc_event - departure
except TypeError as error:
    print(type(error).__name__, str(error))

TypeError can't subtract offset-naive and offset-aware datetimes


### **Перевод часов и fold.**

При переходе на зимнее время часы переводят назад. В Нью-Йорке 1 ноября 2026 года время `01:30` встречается дважды. Атрибут `fold` различает повторения: `0` – первое, `1` – второе. Для этого примера дата перехода известна из правил `IANA`.

Покажем обе записи через `.isoformat()`: этот метод создаёт строку с датой, временем и смещением, если оно задано.

In [89]:
first_0130 = datetime(2026, 11, 1, 1, 30, tzinfo=new_york, fold=0)
second_0130 = datetime(2026, 11, 1, 1, 30, tzinfo=new_york, fold=1)
first_0130.isoformat(), second_0130.isoformat()

('2026-11-01T01:30:00-04:00', '2026-11-01T01:30:00-05:00')

Метод `.isoformat()` вернул стандартное строковое представление с разными смещениями. Переведём оба значения в `UTC` перед вычитанием: получим фактически прошедшую длительность между двумя событиями.

In [90]:
second_0130.astimezone(UTC) - first_0130.astimezone(UTC)

datetime.timedelta(seconds=3600)

**Тонкость для внимательного разбора.** При вычитании двух aware-объектов `Python` `datetime` с одним и тем же объектом `tzinfo` пояс и `fold` не участвуют в разности: сравниваются местные календарные показания. Поэтому прямое вычитание здесь даст ноль. Для прошедшего времени мы явно привели оба события к `UTC`.

In [91]:
second_0130 - first_0130

datetime.timedelta(0)

Весной возникает обратная ситуация: некоторые местные часы пропускаются. Например, `02:30` в Нью-Йорке 8 марта 2026 года не существовало. Можно создать такой `datetime` конструктором, но сам факт создания не доказывает, что это реальный местный момент. В `pandas` есть явная обработка таких случаев; вернёмся к ней после введения временных столбцов.

Для совершившихся событий обычно удобно хранить момент в `UTC`, а местный пояс сохранять для отображения и календарной аналитики. Для будущего расписания нужны также местные часы и имя региона: одна фиксированная `UTC`-метка не описывает правило «каждый день в 09:00».

---

### **pytz в старом коде.**

В существующих проектах встречается `pytz` – отдельная библиотека с базой часовых поясов. Для нового кода на нашей версии `Python` используем `zoneinfo`; авторы `pytz` тоже рекомендуют его для `Python` 3.9 и новее.

Два следующих примера нужны, чтобы узнавать старый API. Для них установите пакет в окружение ноутбука:

```bash
python -m pip install pytz
```

У `pytz` местное naive-время связывают с регионом методом `localize()`. Не переносим сюда приём `datetime(..., tzinfo=...)` с региональным объектом `pytz`: он может дать историческое смещение вместо ожидаемого.

In [92]:
import pytz

legacy_new_york = pytz.timezone("America/New_York")
legacy_noon = legacy_new_york.localize(datetime(2026, 1, 15, 12), is_dst=None)
legacy_noon.astimezone(UTC), legacy_noon == winter_noon

(datetime.datetime(2026, 1, 15, 17, 0, tzinfo=datetime.timezone.utc), True)

`is_dst=None` требует сообщить об ошибке для неоднозначного или несуществующего местного времени. Возьмём уже знакомый повторный час: метод не выбирает повторение молча.

In [93]:
try:
    legacy_new_york.localize(datetime(2026, 11, 1, 1, 30), is_dst=None)
except pytz.AmbiguousTimeError:
    print("Нужно уточнить, какое повторение часа имеется в виду")

Нужно уточнить, какое повторение часа имеется в виду


У двух API разные правила, поэтому механической замены импорта недостаточно:

| Операция | `zoneinfo` | `pytz` |
| --- | --- | --- |
| Создание местного значения | `tzinfo=ZoneInfo(...)` | `localize(...)` |
| Различение повторного часа | `fold` | Параметр `is_dst` у `localize` |
| Перевод известного момента | `astimezone(...)` | `astimezone(...)` |

После арифметики через переход смещения в старом коде `pytz` может потребоваться `normalize()`, чтобы пересчитать местные показания и смещение. Параметр `is_dst` выбирает сторону сезонного перехода и не является универсальным аналогом `fold` для всех изменений времени. [Правила API pytz](https://pypi.org/project/pytz/).

Дальше работаем с уже знакомыми объектами `ZoneInfo`.

## **Время в логах.**

**Какой пояс встретится в логах компании?** Единого формата для всех компаний нет. Для объединения событий разных сервисов распространённая практика – нормализовать моменты в `UTC`. Но конкретный лог может содержать локальные часы сервера, явное смещение или числовую метку. Контракт источника проверяем до разбора; местоположение сервера само по себе не определяет формат поля.

В текстовых API и логах часто используют запись по `RFC 3339`: дата, разделитель `T`, время, при необходимости дробная часть секунды, затем `Z` или смещение. `Z` обозначает `UTC`, `+00:00` – нулевое смещение. Обозначение `+03:00` сообщает смещение, но не имя региона `IANA`.

Разберём две записи одного события. `fromisoformat` в `Python` 3.14 понимает эти варианты, хотя множество принимаемых им форматов шире `RFC 3339`.

In [94]:
logged_utc = datetime.fromisoformat("2026-01-15T09:00:00.123Z")
logged_offset = datetime.fromisoformat("2026-01-15T12:00:00.123+03:00")
logged_utc == logged_offset

True

Чтобы записать событие единообразно, сначала переводим его в `UTC`, затем вызываем `isoformat()`. Получится явное `+00:00`. Нельзя просто дописать `Z` к местным часам: это изменит смысл записи.

In [95]:
logged_offset.astimezone(UTC).isoformat()

'2026-01-15T09:00:00.123000+00:00'

Не всякий лог хранит строку. Например, модель логов OpenTelemetry задаёт временные метки целым числом наносекунд от `Unix Epoch`. В других API единицей могут быть секунды или миллисекунды. Поэтому при получении числа выясняем **единицу, эпоху и смысл поля**, а не угадываем их только по длине числа.

Описание текстового формата: [`RFC 3339`](https://www.rfc-editor.org/rfc/rfc3339). Пример конкретной схемы телеметрии: [OpenTelemetry Logs Data Model](https://opentelemetry.io/docs/specs/otel/logs/data-model/).

### **Время события и получения.**

Ошибка произошла в 09:00:00, а сборщик получил запись в 09:00:03. Это два момента. По первому строим отчёт о событиях, по разности оцениваем задержку получения.

Названия полей зависят от системы: например, OpenTelemetry различает `Timestamp` и `ObservedTimestamp`. В нашем учебном наборе используем `event_at` и `received_at`; перед работой с настоящими логами проверяем смысл полей в схеме источника.

### **Синхронизация часов.**

Два сервера в `UTC` могут показывать разное время: общая шкала не исправляет отставание часов. Для синхронизации используют, например, NTP – протокол сверки времени по сети. Поэтому отрицательная задержка между метками разных машин может означать рассогласование часов или ошибку данных.

Связанные записи разных сервисов находят по идентификатору запроса или трассы. Сортировка только по `timestamp` не доказывает причинный порядок. Для измерения длительности операции на одной машине есть отдельные часы.

### **Монотонные часы.**

Системное календарное время может корректироваться при синхронизации. Для измерения длительности выполнения кода есть монотонные часы: `perf_counter()` из модуля `time`. Их начало отсчёта не имеет календарного смысла; используем только разность двух показаний. Импортируем функцию отдельно, чтобы не заменить уже знакомое имя класса `time`.

In [96]:
from time import perf_counter

started = perf_counter()
number_total = sum(range(10_000))
elapsed_seconds = perf_counter() - started
elapsed_seconds >= 0

True

Мы проверили только неотрицательность длительности, а не сравнивали скорость библиотек: для такого сравнения нужны повторения и контролируемые условия. Unix `timestamp` отвечает «когда произошло», `perf_counter` – «сколько заняло выполнение».

---

Теперь перенесём разобранные операции на столбцы `Pandas`: преобразуем строки, проверим ошибки дат и вычислим длительности.

---

## **Время в `Pandas`.**

`pd.to_datetime` превращает **весь столбец** строк в значения времени. Параметр `format` описывает порядок частей: `%Y` – год, `%m` – месяц, `%d` – день; затем часы, минуты и секунды. Укажем его явно, чтобы не полагаться на догадку при чтении.

In [97]:
taxi["pickup"] = pd.to_datetime(taxi["pickup"], format="%Y-%m-%d %H:%M:%S")
taxi["pickup"].head(3)

0   2019-03-23 20:21:09
1   2019-03-04 16:11:55
2   2019-03-27 17:53:01
Name: pickup, dtype: datetime64[us]

Теперь в столбце хранятся календарные даты со временем типа `datetime64`; отдельное значение представлено объектом `Timestamp`. Пояс ещё не задан: это местные показания часов, которые сами по себе не определяют однозначный момент на общей шкале.

In [98]:
taxi["pickup"].dtype, type(taxi.loc[0, "pickup"])

(dtype('<M8[us]'), pandas.Timestamp)

### **Тип `Timestamp`.**

Unix `timestamp` – число с согласованной единицей и эпохой. `pd.Timestamp` – объект `pandas` для одного значения даты со временем. Он может хранить пояс и поддерживает календарные операции. Имя класса не означает, что любое переданное число автоматически измеряется в секундах.

В `pandas` значение времени представляется числом отсчётов относительно эпохи вместе с единицей разрешения. У временного столбца с поясом значения привязаны к `UTC`, а пояс задаёт местное представление. Для naive-столбца такого однозначного соответствия реальному моменту нет. Тип может иметь разные единицы, например `datetime64[us]` (микросекунды) или `datetime64[ns]` (наносекунды); не следует считать наносекунды универсальным внутренним форматом.

Разберём ранее полученное целое число миллисекунд: явно зададим `unit="ms"`, `origin="unix"` и `utc=True`.

In [99]:
numeric_times = pd.Series([event_milliseconds, event_milliseconds + 1000])
numeric_dates = pd.to_datetime(numeric_times, unit="ms", origin="unix", utc=True)
numeric_dates

0   2023-11-14 22:13:20.123+00:00
1   2023-11-14 22:13:21.123+00:00
dtype: datetime64[us, UTC]

Разность строк – одна секунда, хотя исходные числа отличаются на 1000. Если выбрать другую единицу, получится другая дата либо ошибка диапазона. Для Unix-числа `utc=True` задаёт правильную исходную шкалу; для местного текста без смещения этот параметр означал бы предположение «текст уже описывает `UTC`», которое нужно обосновать.

Чем мельче единица при фиксированном размере целого, тем меньше диапазон дат. Поэтому разрешение и диапазон связаны. Для одного значения используйте `pd.Timestamp`, для разбора столбца – `pd.to_datetime`; не вызывайте последнее отдельно для каждой строки в `Python`-цикле.

В следующей ячейке `.diff()` вычитает из каждого значения предыдущее, а `.iloc[1]` выбирает вторую позицию – первую, у которой есть предшественник.

In [100]:
pd.Timestamp(event_milliseconds, unit="ms", tz="UTC"), numeric_dates.diff().iloc[1]

(Timestamp('2023-11-14 22:13:20.123000+0000', tz='UTC'),
 Timedelta('0 days 00:00:01'))

### **Наносекунды и потери точности.**

Возьмём целые Unix-наносекунды и соседний отсчёт. `unit="ns"` сообщает единицу; `utc=True` делает временную шкалу явной. Не переводим большие числа через `float`.

In [101]:
base_ns = 1_700_000_000_123_456_789
fine_events = pd.to_datetime(pd.Series([base_ns, base_ns + 1]), unit="ns", utc=True)
fine_events

0   2023-11-14 22:13:20.123456789+00:00
1   2023-11-14 22:13:20.123456790+00:00
dtype: datetime64[ns, UTC]

Вычтем первое событие из второго. Разница существует, хотя оба произошли внутри одной микросекунды.

In [102]:
fine_events.iloc[1] - fine_events.iloc[0]

Timedelta('0 days 00:00:00.000000001')

У `Timestamp` атрибут `.microsecond` хранит микросекундную часть, а `.nanosecond` – остаток наносекунд внутри микросекунды, от 0 до 999. Это не полное число наносекунд от эпохи.

In [103]:
fine_events.iloc[0].microsecond, fine_events.iloc[0].nanosecond

(123456, 789)

Обычный `Python` `datetime` имеет микросекундное разрешение. Метод `.to_pydatetime()` поэтому отбрасывает наносекундный остаток и обычно предупреждает об этом. Здесь намеренно задаём `warn=False` только для демонстрации заранее объяснённой потери. Сравним два результата.

In [104]:
coarse_first = fine_events.iloc[0].to_pydatetime(warn=False)
coarse_second = fine_events.iloc[1].to_pydatetime(warn=False)
coarse_first, coarse_first == coarse_second

(datetime.datetime(2023, 11, 14, 22, 13, 20, 123456, tzinfo=datetime.timezone.utc),
 True)

События стали неразличимы. Если порядок важен, сохраняем исходную точность или отдельный порядковый номер. Теперь проверим ещё один способ потерять различие: переведём соседние большие целые числа во `float`.

In [105]:
float(base_ns) == float(base_ns + 1)

True

Для этих чисел ответ `True`: расстояние между представимыми значениями `float` уже больше единицы. Потеря произошла ещё до создания даты. При обмене данными проверяют всю цепочку: источник, транспорт, парсер, тип столбца и сохранение. Поддержка наносекунд одной библиотекой не гарантирует их сохранность на остальных этапах.

[Преобразование в `Python` `datetime`](https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.to_pydatetime.html) документирует потерю наносекундного остатка.

### **Часовые пояса в `Pandas`.**

В `Python` мы различили назначение пояса и перевод момента. У временного столбца доступ `.dt` открывает операции над датами и длительностями, подобно `.str` для текста. Назначению пояса и переводу соответствуют `.dt.tz_localize()` и `.dt.tz_convert()`. Создадим отдельно учебные местные часы с известным источником, не назначая пояс исходным поездкам наугад.

In [106]:
known_local_times = pd.to_datetime(pd.Series(["2026-01-15 12:00", "2026-07-15 12:00"]), format="%Y-%m-%d %H:%M")
ny_times = known_local_times.dt.tz_localize("America/New_York")
ny_times.dt.tz_convert("UTC")

0   2026-01-15 17:00:00+00:00
1   2026-07-15 16:00:00+00:00
dtype: datetime64[us, UTC]

При локализации `pandas` по умолчанию сообщает об ошибке, если местное время неоднозначно или не существует. Можно явно выбрать политику. `NaT` – пропуск временного типа: дата или момент не определены. `ambiguous="NaT"` отмечает повторяющиеся часы как неопределённые, `nonexistent="NaT"` – пропущенные часы. Это сохраняет проблему видимой, пока не уточнены данные источника.

In [107]:
transition_times = pd.to_datetime(pd.Series([
    "2026-03-08 02:30", "2026-11-01 01:30", "2026-11-01 03:30",
]), format="%Y-%m-%d %H:%M")
localized_transitions = transition_times.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT",
)
localized_transitions

0                         NaT
1                         NaT
2   2026-11-01 03:30:00-05:00
dtype: datetime64[us, America/New_York]

Первые две записи требуют дополнительного решения, третья однозначна. Если бизнес-правило разрешает переносить несуществующее время вперёд, есть `nonexistent="shift_forward"`, но это изменение исходных часов, а не восстановление неизвестного события. Для повторного часа нужно выяснить, какое из двух событий имелось в виду.

Теперь вернёмся к поездкам: их исходные часы по-прежнему остаются без назначенного пояса.

Вернёмся к явному формату `%Y-%m-%d %H:%M:%S` из `CSV` и преобразуем время высадки. Разность двух столбцов даст длительность по записанным местным часам.

In [108]:
taxi["dropoff"] = pd.to_datetime(taxi["dropoff"], format="%Y-%m-%d %H:%M:%S")
taxi[["pickup", "dropoff"]].head(3)

,pickup,dropoff
0,2019-03-23 20:21:09,2019-03-23 20:27:24
1,2019-03-04 16:11:55,2019-03-04 16:19:00
2,2019-03-27 17:53:01,2019-03-27 18:00:25


Какие даты вообще покрывает выборка? Здесь `min` и `max` ищут самый ранний и самый поздний момент посадки.

In [109]:
taxi["pickup"].min(), taxi["pickup"].max()

(Timestamp('2019-02-28 23:29:03'), Timestamp('2019-03-31 23:43:45'))

Получили границы записей в файле. Они описывают нашу выборку, но не доказывают, что внутри неё есть все поездки за какой-либо день. Проверим пропуски в датах.

In [110]:
taxi[["pickup", "dropoff"]].isna().sum()

pickup     0
dropoff    0
dtype: int64

В исходном файле обе даты заполнены. Но в учебных квитанциях мы уже встретили невозможную дату и пропуски. По умолчанию `pd.to_datetime` сообщает об ошибке разбора. С `errors="coerce"` неразбираемые значения становятся `NaT` – пропуском временного типа. Преобразуем весь `date_texts`, извлечённый регуляркой.

In [111]:
parsed_receipt_dates = pd.to_datetime(date_texts, format="%d.%m.%Y", errors="coerce")
parsed_receipt_dates

0   2019-03-15
1          NaT
2          NaT
3          NaT
Name: receipt, dtype: datetime64[us]

`NaT` получился и там, где дата невозможна, и там, где не было извлечённого текста. Не будем терять это различие. `.notna()` отмечает заполненные значения; совместим эту маску с `.isna()` результата.

In [112]:
invalid_date = date_texts.notna() & parsed_receipt_dates.isna()
date_texts.loc[invalid_date]

1    31.02.2019
Name: receipt, dtype: string

Так мы нашли именно ошибку преобразования: текст был, а даты из него не получилось. `errors="coerce"` полезен вместе с проверкой качества, а не вместо неё.

---

## **Длительность поездки.**

Временные столбцы уже преобразованы. Вычтем посадку из высадки: получим `Timedelta`, затем переведём длительность в минуты. `.dt.total_seconds()` учитывает весь интервал, включая дни и дробную часть секунды.

In [113]:
taxi["duration"] = taxi["dropoff"] - taxi["pickup"]
taxi["duration_min"] = taxi["duration"].dt.total_seconds() / 60

taxi[["pickup", "dropoff", "duration_min"]].head(3)

,pickup,dropoff,duration_min
0,2019-03-23 20:21:09,2019-03-23 20:27:24,6.250000
1,2019-03-04 16:11:55,2019-03-04 16:19:00,7.083333
2,2019-03-27 17:53:01,2019-03-27 18:00:25,7.400000


Вычитаем полные даты, поэтому переход через полночь учитывается автоматически. Но исходные часы не содержат пояса: возле сезонного перевода их разность может отличаться от фактически прошедшего времени.

### **События за местный день.**

В логах момент часто записан в `UTC`. Найдём события **за 16 января по Москве**. Сначала разберём метки с `Z` через `utc=True`, затем покажем те же моменты в `Europe/Moscow`. Формат `ISO8601` принимает такие записи.

In [114]:
service_events = pd.DataFrame(
    {
        "request_id": ["AB-204", "CD-120"],
        "event_at": ["2026-01-15T21:30:00Z", "2026-01-16T21:00:00Z"],
    }
)
service_events["event_at"] = pd.to_datetime(
    service_events["event_at"],
    format="ISO8601",
    utc=True,
)
service_events["event_local"] = service_events["event_at"].dt.tz_convert("Europe/Moscow")

service_events

,request_id,event_at,event_local
0,AB-204,2026-01-15 21:30:00+00:00,2026-01-16 00:30:00+03:00
1,CD-120,2026-01-16 21:00:00+00:00,2026-01-17 00:00:00+03:00


Первое событие произошло 16 января в 00:30 по Москве, второе – уже 17 января в 00:00. Выберем интервал **от начала 16 января включительно до начала 17 января исключительно**: `[начало, конец)`. Так полночь не попадёт сразу в два дневных отчёта.

Задаём обе местные полуночи явно: в регионах с переводом часов календарный день не всегда равен 24 часам.

In [115]:
report_start = pd.Timestamp("2026-01-16", tz="Europe/Moscow")
report_stop = pd.Timestamp("2026-01-17", tz="Europe/Moscow")

# Правая граница принадлежит следующему дню.
in_report_day = (service_events["event_local"] >= report_start) & (
    service_events["event_local"] < report_stop
)
service_events.loc[in_report_day, ["request_id", "event_local"]]

,request_id,event_local
0,AB-204,2026-01-16 00:30:00+03:00


### **Дневная сводка.**

Вернёмся к поездкам. `.dt.floor("D")` переносит записанное время на полночь той же даты, сохраняя временной тип. Сгруппируем строки по этим датам: посчитаем поездки и медианную стоимость.

In [116]:
taxi["pickup_day"] = taxi["pickup"].dt.floor("D")
daily_summary = taxi.groupby("pickup_day").agg(
    trips=("fare", "size"),
    median_fare=("fare", "median"),
)
daily_summary.head()

,trips,median_fare
pickup_day,,
2019-02-28,1,5.0
2019-03-01,241,9.0
2019-03-02,198,9.0
2019-03-03,169,9.5
2019-03-04,171,9.5


Теперь одна строка соответствует дню. Сводка описывает учебную выборку: отсутствие записей за день не означает отсутствие поездок во всём городе.

---

## **Дополнительное чтение.**

Не обязательно читать всё. Выберите материал по теме, которую хотите закрепить:

- [`Regex HOWTO`](https://docs.python.org/3.14/howto/regex.html) – последовательное объяснение шаблонов, поиска и жадности.
- [Модуль `datetime`](https://docs.python.org/3.14/library/datetime.html) – календарные типы, длительности и арифметика времени.
- [Текстовые столбцы `Pandas`](https://pandas.pydata.org/docs/user_guide/text.html) – строковые операции и регулярные выражения через `.str`.
- [Время в `Pandas`](https://pandas.pydata.org/docs/user_guide/timeseries.html) – преобразования, пояса, индексы и временные интервалы.

Для расширения кругозора: [`UTC` is Enough for Everyone, Right?](https://zachholman.com/talk/utc-is-enough-for-everyone-right) – почему момент события, календарная дата и расписание требуют разных моделей.